# 2. ERA5: model evaluation and observed change

Two questions, both asked of ERA5 and the LESFMIP ensembles over the years they share (1979–2014):

1. **Is each model consistent with ERA5?** If ERA5 could be one more member of a model's historical ensemble, the model simulates both the forced response (the signal) and the internal variability (the noise) well enough to take its single-forcing experiments seriously. That is part 1.
2. **Is the observed change detectable, and which forcings is it consistent with?** ERA5 against hist-nat (could it have happened without human influence?) and against each single forcing, then scaling factors. That is part 2.

The data come from notebook 01. Run either *Open the data* section, then everything below in order.

Each step prints what it made, with a **quick look** (a plain xarray plot, all its code here) before the method figures. Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from dask.distributed import wait
from plotting_modules import formatting, maps
from extant import config, loading, paths, preprocessing as pp, storage
from extant.logger import setup_logging
from extant.plots import methods

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Dask warns about every large task graph; the graphs here are large on purpose
warnings.filterwarnings('ignore', message='.*Sending large graph of size.*', category=UserWarning)

from extant import datatree, era5_evaluation as ev, observed_change as oc
from extant.plots import era5_evaluation as evp, observed_change as ocp, response_change as rcp

## Settings

What to evaluate, and where. `RUN` is set by whichever *Open the data* section runs: results from the sample are saved apart from those from the full data.

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]

EVAL_EXPERIMENT = 'historical'                        # all forcings: the only experiment ERA5 should resemble
EVAL_POINT = config.POINT                             # the grid point used throughout (the nearest to it)
EVAL_YEARS = slice(config.ERA5_START_YEAR, 2014)      # ERA5 is used from 1979; LESFMIP ends with DJF 2013
EVAL_MODEL = config.PLOT_SEL['model']                 # model shown in the single-model figures
EVAL_SEASON = 'DJF'                                   # season shown in the single-season figures


def result_path(name):
    """Where a result of this notebook is saved: results/<variable>/<run>/<name>.zarr."""
    return paths.result(name, VARIABLE, RUN)

## Open the data

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (seasonal LESFMIP means as a DataTree, `/<model>/<experiment>`, on member × year × season × lat × lon), `era5_seasonal_da` (ERA5 on year × season × lat × lon, on the same grid) and `RUN`, all in analysis units.

### Sample data

The 3×3 block of grid cells around `config.POINT` in `tests/data` (two models), put through the same pre-processing as the full data in notebook 01. Everything below runs in a few minutes, with maps of only 3×3 cells.

In [ ]:
lesfmip_season_tree = pp.seasonal_tree(loading.open_lesfmip_sample(experiments=config.MAIN_EXPERIMENTS), VARIABLE).compute()
era5_seasonal_da = pp.seasonal_era5(loading.open_era5_sample(), VARIABLE, like=lesfmip_season_tree)
RUN = 'sample'

### Full data (JASMIN)

The seasonal means saved by notebook 01 (section 3): `seasonal/lesfmip_<variable>.zarr` and `seasonal/era5_<variable>.zarr`, read from `paths.DATA_DIR` if they have been moved there and otherwise from `paths.SCRATCH`. Opening is lazy; each section below loads only what it needs.

In [ ]:
#(t): DATA_DIR is read first, then SCRATCH
seasonal_path = paths.find(paths.seasonal('lesfmip', VARIABLE))
era5_path = paths.find(paths.seasonal('era5', VARIABLE))
print(f'reading {seasonal_path}')
print(f'reading {era5_path}')
lesfmip_season_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})
era5_seasonal_da = xr.open_zarr(era5_path)[VARIABLE].load()

#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_season_tree.filter(lambda node: node.name in config.MAIN_EXPERIMENTS)
RUN = 'full'

### What was opened

Whichever section ran: every model and experiment in `lesfmip_season_tree`, and ERA5.

In [ ]:
#(t): Every model and experiment: its members, years, seasons and grid
pd.DataFrame({
    node.path: {'members': node.sizes['member'], 'years': f'{int(node.year[0])}-{int(node.year[-1])}',
                'seasons': ', '.join(node.season.values), 'lat x lon': f"{node.sizes['lat']} x {node.sizes['lon']}"}
    for node in lesfmip_season_tree.leaves
}).T

In [ ]:
#(t): ERA5's seasonal means, on (year, season, lat, lon)
print(f'results are saved under {paths.SCRATCH / "results" / VARIABLE / RUN}')
era5_seasonal_da

## A first look

Before any test: ERA5 and the models' members at the evaluation point, and the 1979–2014 mean of ERA5 next to each model's ensemble mean. The maps show the mean-state bias, which only the *raw values* tests see; every other test removes it.

In [ ]:
#(t): ERA5's seasonal means at the point
evp.order_seasons(era5_seasonal_da.sel(**EVAL_POINT)).plot(hue='season')
plt.title(f'ERA5 at {formatting.format_latlon(EVAL_POINT)}');

In [ ]:
#(t): Each experiment's members at the point (one model, DJF)
fig, axes = methods.members_by_experiment(lesfmip_season_tree[EVAL_MODEL].sel(**EVAL_POINT).load(),
                                          config.MAIN_EXPERIMENTS, VARIABLE, units=var.units)

In [ ]:
#(t): 1979-2014 mean, DJF: ERA5, and each model's ensemble mean minus ERA5, on (model, lat, lon)
era5_mean_da = era5_seasonal_da.sel(year=EVAL_YEARS, season=EVAL_SEASON).mean('year')
model_bias_da = datatree.reduce_to_dataset(
    lesfmip_season_tree.match(f'*/{EVAL_EXPERIMENT}'),
    lambda ds: ds.sel(year=EVAL_YEARS, season=EVAL_SEASON).mean(['year', 'member']),
).sel(experiment=EVAL_EXPERIMENT, drop=True)[VARIABLE].load() - era5_mean_da
print(model_bias_da)

In [ ]:
#(t): Quick look: each model's bias on the plain lat-lon grid
model_bias_da.plot(x='lon', y='lat', col='model', cmap='RdBu_r', center=0);

In [ ]:
panels = maps.polar_grid(era5_mean_da, levels=np.linspace(float(era5_mean_da.min()), float(era5_mean_da.max()), 15),
                         cmap='RdYlBu_r', title=f'ERA5, {EVAL_SEASON} 1979-2014', cbar_label=var.label, tag=False)
panels = maps.polar_grid(model_bias_da, col_dim='model', levels=np.linspace(-5, 5, 11), cmap='RdBu_r',
                         title=f'Model minus ERA5, {EVAL_SEASON} 1979-2014 (ensemble mean)',
                         cbar_label=f'Bias ({var.units})', tag=False)

# Part 1: is each model consistent with ERA5?

**Is each model's historical ensemble consistent with ERA5, locally?** Every test in this section asks one question: *could ERA5 be one more member of the ensemble?* If a model simulates both the forced response (the **signal**) and the internal variability (the **noise**) correctly, ERA5 is statistically indistinguishable from any single member. So each test computes the same quantity for ERA5 and for every member, and asks where ERA5 falls among the members.

The members are the null distribution. That means no assumption about the shape of the distribution, and the short record (1979–2014, ~35 values per season) is accounted for automatically, because each member is exactly as long as ERA5. This is the framework of Suarez-Gutierrez, Milinski & Maher (2021), *Exploiting large ensembles for a better yet simpler climate model evaluation*, Clim. Dyn., [doi:10.1007/s00382-021-05821-w](https://doi.org/10.1007/s00382-021-05821-w).

**Three treatments.** ERA5 and every member are treated identically, and each treatment isolates a different question:

| Treatment | Removed from each series | A comparison then tests |
|---|---|---|
| Raw values | nothing | the climatology; any mean-state bias dominates |
| Anomalies | its own 1979–2014 mean | the forced trend (signal) and the variability (noise) |
| Detrended anomalies | its own least-squares line | the variability (noise) only |

**The tests.**

| Test | Statistic | Treatment | ERA5 low / high means |
|---|---|---|---|
| Moment | mean | raw | model too warm / too cold |
| Moment | standard deviation | detrended | model too variable / not variable enough (**noise**) |
| Moment | skewness | detrended | model skewness too high / too low |
| Moment | excess kurtosis | detrended | model tails too heavy / too light |
| Other statistic | trend | raw | model trend too large / too small (**signal**) |
| Other statistic | S/N against itself: smoothed change over the record / std of the deviations from it | raw | model emerges too strongly / too weakly (**signal**) |
| Other statistic | width, Q95 − Q05 | detrended | model distribution too wide / too narrow |
| Other statistic | cold tail Q50 − Q05, warm tail Q95 − Q50 | detrended | that tail too long / too short |
| Other statistic | number of record highs and lows | raw | model sets too many / too few records |
| Other statistic | lag-1 autocorrelation | detrended | model too persistent / not persistent enough |
| Rank histogram | dispersion (dome / U shape) | anomalies, detrended | ensemble spread too large / too small |
| Rank histogram | rank trend | anomalies | model warms faster / slower than ERA5 (**signal**) |
| Plume | % of seasons outside the ensemble range | raw, anomalies | compared with 200 / (N + 1) % |
| Spatial evaluation | % of years below every member, above every member, and in the members' central 75% | anomalies | ≥ 10%, ≥ 10% and > 80% (Suarez-Gutierrez et al., 2021) |

**The moment test** is the first four rows (`ev.moment_test`): ERA5's mean, standard deviation, skewness and excess kurtosis, each computed over the same ~35 years for ERA5 and for every member. The standard deviation, skewness and kurtosis are of detrended values, so the 1979–2014 warming neither inflates the spread nor bends the shape. The other statistics (`ev.statistic_test`) are not moments, but are tested in exactly the same way.

**Why this matters for the response figures.** The rest of the notebook attributes two things to each forcing: the temperature *response* (the shift of the median, tested with the t-test and S/N), and changes in *variability* (the Q95 − Q05 width and whether the cold or warm tail changes faster). A model's response is only credible where it gets the observed signal right, and its variability change only where it gets the observed noise, width and tails right. So the signal statistics (trend, rank trend) check the first, and the noise statistics (std, width, both tails, rank dispersion) check the second, one model and one season at a time.

**Reading the results.** Every test reports **ERA5's percentile among the members** (50 = centred; 0 or 100 = beyond every member) and a two-sided p-value. With `ev.ALPHA = 0.1`, a statistic is flagged (✗) when ERA5 falls outside the members' 5–95% range, so a *perfect* model is still flagged about 10% of the time by chance. With N members the smallest possible p-value is 2 / (N + 1), so an ensemble needs at least 20 members to show anything.

**Code.** Calculations live in `extant/era5_evaluation.py` (imported as `ev`) and figures in `extant/plots/era5_evaluation.py` (`evp`); every function there is documented. Everything runs in this notebook process on data loaded at one point (or, in sections 7, 8 and part 2's detection maps, one model's map at a time), so no cluster is needed. Those three map calculations take minutes, so they save their results; each has a *reopen* cell to run instead next time.

**Contents**
- Setup: ERA5 to seasonal means, aligning each model with ERA5, running every test
- 1\. How the tests work: toy ensembles with a known answer
- 2\. Distributions: do the raw values fit? do the anomalies? (KDE and Q–Q)
- 3\. Time series: is ERA5 inside or outside the ensemble plume?
- 4\. The moment test, and the other statistics
- 5\. Rank histograms (Suarez-Gutierrez et al., 2021)
- 6\. All models: scorecard and signal-vs-noise diagram
- 7\. Maps: the moment test at every grid point, and whether each model is acceptable over the whole map
- 8\. The spatial evaluation of Suarez-Gutierrez et al. (2021): where each model is adequate, and if not, why
- Caveats

## Setup

### ERA5 on the model grid

ERA5 was turned into seasonal means from its monthly means (from 1979, on its own standard calendar). `pp.seasonal_era5` (run in notebook 01, or in the sample section above) follows the LESFMIP pre-processing step for step (`QS-DEC` bins, complete seasons and whole years only, then `split_time`), so DJF is labelled by the year of its December in both and the two line up label for label, whatever their calendars. `pp.match_grid` then puts ERA5 on the model grid: the same longitude convention (0–360 or −180–180), and the model's exact points, taken unchanged where the two grids share them and bilinearly interpolated where they do not.

`lesfmip_obs_tree` is LESFMIP over the same years as ERA5 (`EVAL_YEARS`), the input to every comparison with ERA5 below.

In [ ]:
#(t): LESFMIP over the years that match ERA5
lesfmip_obs_tree = lesfmip_season_tree.sel(year=EVAL_YEARS)

### Aligning each model with ERA5

Each test compares a statistic of ERA5 with the same statistic of every member, which is only fair if all of them come from exactly the same samples. `ev.align` keeps the years both records cover, drops members with no data, and masks any year-season missing from ERA5 or from any member in *all* of them. For example, LESFMIP ends with DJF 2013 (December 2013 to February 2014), so ERA5's later seasons are left out.

The table lists what each model is compared on. The last column is how often ERA5 would fall outside the ensemble range by chance alone if it were a member.

In [ ]:
#(t): One (ensemble, ERA5) pair per model with EVAL_EXPERIMENT, at EVAL_POINT: same years, same missing-data mask
eval_pairs = {
    node.parent.name: ev.align(node[VARIABLE].sel(**EVAL_POINT).load(), era5_seasonal_da.sel(**EVAL_POINT), years=EVAL_YEARS)
    for node in lesfmip_obs_tree.match(f'*/{EVAL_EXPERIMENT}').leaves
}

#(t): The grid point actually used, for figure titles
point_obs_da = eval_pairs[EVAL_MODEL][1]
POINT_LABEL = formatting.format_latlon({'lat': float(point_obs_da.lat), 'lon': float(point_obs_da.lon)})

#(c): N < 20 cannot give p < 0.1
pd.DataFrame({
    model: {
        'members (N)': ensemble.sizes['member'],
        'years': f'{int(obs.year.min())}-{int(obs.year.max())}',
        'seasons compared': int(obs.count()),
        'outside range by chance (%)': round(200 / (ensemble.sizes['member'] + 1), 1),
    }
    for model, (ensemble, obs) in eval_pairs.items()
}).T

In [ ]:
#(t): What one pair holds: EVAL_MODEL's ensemble on (member, year, season), and ERA5 on (year, season), at the point
eval_ensemble_da, eval_obs_da = eval_pairs[EVAL_MODEL]
print(eval_ensemble_da, '\n')
print(eval_obs_da)

In [ ]:
#(t): Quick look: ERA5 (black) among EVAL_MODEL's members (grey), every season
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharex=True)
for ax, season in zip(axes, ['DJF', 'MAM', 'JJA', 'SON']):
    ax.plot(eval_ensemble_da.year, eval_ensemble_da.sel(season=season).transpose('year', 'member'), color='0.7', lw=0.5)
    ax.plot(eval_obs_da.year, eval_obs_da.sel(season=season), color='k')
    ax.set_title(season)

In [ ]:
#(t): hist-nat at the same point, as context for the time-series figures
#(c): ERA5 should follow historical, not hist-nat; where the two plumes differ, the figures show which it follows
histnat_plumes = {
    node.parent.name: ev.ensemble_plume(*ev.align(node[VARIABLE].sel(**EVAL_POINT).load(), era5_seasonal_da.sel(**EVAL_POINT),
                                                  years=EVAL_YEARS), treatment='anomaly')
    for node in lesfmip_obs_tree.match('*/hist-nat').leaves if node.parent.name in eval_pairs
}

### Running every test

`ev.evaluate` runs everything for one model and returns a dict:
- `moments`: the moment test (mean, standard deviation, skewness, excess kurtosis), per season
- `statistics`: the other statistics (trend, S/N, width, tails, records, lag-1), tested the same way, per season
- `ranks`: rank-histogram tests with the four seasons pooled (4x the samples), for raw values, anomalies and detrended anomalies
- `ranks_seasonal`: rank-histogram tests per season (for the scorecard)
- `plumes`: ensemble quantiles through time, and where ERA5 sits relative to them
- `distributions`: KDEs and quantiles

`ev.summary_table` stacks the per-season results of every model into one `(model, statistic, season)` Dataset.

In [ ]:
%%time
#(t): Every test for every model at the point (seconds: ~50 members x 35 years x 4 seasons)
eval_results = {model: ev.evaluate(ensemble, obs) for model, (ensemble, obs) in eval_pairs.items()}
eval_table_ds = ev.summary_table(eval_results)

In [ ]:
#(t): What one model's results hold: a Dataset (or a dict of them) per test; printed, the moment test
print(list(eval_results[EVAL_MODEL]))
print(eval_results[EVAL_MODEL]['moments'])

## 1. How the tests work: toy ensembles with a known answer

Before trusting the tests on real data, here they are on synthetic ensembles where we *know* what is wrong. `ev.simulate_ensemble` makes 50 members and a pseudo-ERA5, 1979–2013, four seasons, each series `trend × decades + noise × N(0, 1)`. Pseudo-ERA5 always has a trend of 0.3 per decade and noise 1; each scenario (`ev.SCENARIOS`) changes one thing about the model:

| Scenario | The model | What the tests should say |
|---|---|---|
| Consistent | identical to ERA5 | nothing flagged, apart from ~10% by chance |
| Too little noise | noise 0.6x | ERA5 often outside the plume; U-shaped histograms; std flagged high |
| Too much noise | noise 1.6x | ERA5 never leaves the plume; dome-shaped histograms; std flagged low |
| Signal too strong | trend 0.9/decade | U shape for anomalies but **flat once detrended**; trend and rank trend flagged |
| Mean-state bias | 2° too warm | raw plume offset; only the mean flagged; every anomaly test passes |

Columns: the raw time series with the plume; the rank histograms of anomalies and of detrended anomalies (bars: pseudo-ERA5; grey bars: the range a perfect model gives); ERA5's percentile for each statistic (✓ inside the grey 5–95% band, ✗ outside).

In [ ]:
#(t): The tests on toy ensembles where the right answer is known
demo_results = {
    name: ev.evaluate(*ev.align(*ev.simulate_ensemble(seed=3, **kwargs)))
    for name, kwargs in ev.SCENARIOS.items()
}
panels = evp.scenario_grid(demo_results, season=EVAL_SEASON, title='The tests on toy ensembles with a known answer')

**Reading the figure**
- **Signal vs noise.** Compare the two rank-histogram columns. A noise problem (rows 2–3) shows in both. A signal problem (row 4) shows only for anomalies: a trend mismatch pushes ERA5 out of the ensemble at both ends of the record, which looks like a U, and detrending removes it.
- **Raw values vs anomalies.** A mean-state bias (row 5) puts ERA5 outside the raw plume much of the time, but once each series is measured from its own mean, every test passes. This is the "raw values don't fit but the anomalies do" case.
- **Too little noise can also flag the mean.** A too-narrow ensemble underestimates how far a 35-year mean can wander. So read the noise tests first.

**Calibration**

Over 200 random toy worlds (40 members), a perfect model is flagged 6–12% of the time for every statistic (nominally 10%). Each fault is caught by its intended test almost every time:

| | noise too small | noise too large | signal too strong | mean bias |
|---|---|---|---|---|
| std (detrended) | 99% | 97% | 10% | 10% |
| width, Q95 − Q05 | 96% | 92% | – | – |
| cold / warm tail | 76% / 77% | 57% / 59% | – | – |
| rank dispersion (anomalies) | 100% | 100% | 68% | 6% |
| rank dispersion (detrended) | 100% | 100% | 8% | 8% |
| trend | 35% | 2% | 96% | 12% |
| rank trend | 21% | 4% | 100% | 9% |
| mean | 30% | 2% | 100% | 100% |

The tail tests are the weakest: with ~35 values per season, each tail rests on about two of them. A ✓ for a tail therefore means "no evidence of a problem", which is weaker than a ✓ for the width or std.

## 2. Distributions: do the raw values fit? do the anomalies?

For each season:
- **Raw values, KDE**: thin blue lines are each member's density, thick blue the pooled ensemble, black ERA5, with ERA5's individual values as a rug underneath. The ✓/✗ is the moment test of the **mean**.
- **Anomalies, KDE**: the same after each series has its own mean removed. The ✓/✗ is the moment test of the **standard deviation** (of detrended anomalies).
- **Anomalies, Q–Q**: the members' quantiles against ERA5's (blue: 5–95% range across members and their median). On the 1:1 line the distributions match. A parallel offset is a bias; steeper than 1:1 means the model is more variable than ERA5, shallower less. Bending away at the ends is a tail (skewness or kurtosis) difference.

ERA5 and every member share one kernel bandwidth (the members' median Scott's-rule bandwidth), so differences between curves come from the data rather than the smoothing.

In [ ]:
#(t): One model, every season
eval_result = eval_results[EVAL_MODEL]
panels = evp.distribution_grid(
    eval_result['distributions'], moments=eval_result['moments'],
    units=var.units, title=f'Distributions: {EVAL_MODEL} {EVAL_EXPERIMENT} vs ERA5 at {POINT_LABEL}',
)

In [ ]:
#(t): Every model, one season
panels = evp.distribution_grid(
    {model: result['distributions'] for model, result in eval_results.items()}, season=EVAL_SEASON,
    moments={model: result['moments'] for model, result in eval_results.items()},
    units=var.units, title=f'Distributions, {EVAL_SEASON}: every model vs ERA5 at {POINT_LABEL}',
)

In [ ]:
#(t): Optional: detrended anomalies instead, to compare the noise alone
panels = evp.distribution_grid(
    eval_result['distributions'], moments=eval_result['moments'],
    columns=(('kde', 'raw'), ('kde', 'detrended'), ('qq', 'detrended')),
    units=var.units, title=f'Distributions (detrended): {EVAL_MODEL} {EVAL_EXPERIMENT} vs ERA5 at {POINT_LABEL}',
)

**What to look for**
- A raw-value KDE shifted from the ensemble is a mean-state bias. If the anomaly KDE then sits on top of the ensemble, the bias was the only problem.
- An anomaly KDE narrower or wider than the members' means the model has too much or too little variability. Check whether the std test (✓/✗) agrees.
- With ~35 values per season, a single ERA5 KDE is noisy. Judge it against the spread of the thin member curves, which have the same sample size, not against the smooth pooled curve.

## 3. Time series: is ERA5 inside or outside the ensemble plume?

Blue bands are the ensemble's central 75% (12.5–87.5%, as in Suarez-Gutierrez et al., 2021), 5–95% and full range each year, with its median. ERA5 is black, with an orange ▲/▼ wherever it is above or below *every* member. If ERA5 were a member, that would happen 200 / (N + 1) % of the time (a few %), as noted in each panel. In the anomaly column, green outlines show the hist-nat ensemble for context.

In [ ]:
#(t): One model: seasons x (raw values, anomalies), with hist-nat (green) for context
histnat_context_ds = histnat_plumes.get(EVAL_MODEL)
panels = evp.plume_grid(
    eval_results[EVAL_MODEL]['plumes'], row_dim='season', col_dim='treatment',
    context=None if histnat_context_ds is None else histnat_context_ds.expand_dims(treatment=['anomaly']),
    units=var.units, title=f'Ensemble plume and ERA5: {EVAL_MODEL} {EVAL_EXPERIMENT} at {POINT_LABEL}',
)

In [ ]:
#(t): Every model's anomaly plume: models x seasons, on one scale
all_plumes_ds = xr.concat(
    [result['plumes'].sel(treatment='anomaly') for result in eval_results.values()],
    dim='model', coords='minimal', compat='override',
).assign_coords(model=list(eval_results))
panels = evp.plume_grid(all_plumes_ds, row_dim='model', col_dim='season', sharey=True, units=var.units,
                        title=f'Anomalies: every model\'s plume and ERA5 at {POINT_LABEL}')

In [ ]:
#(t): % of seasons with ERA5 outside the ensemble range, against chance
outside_da = xr.concat(
    [ev.outside_frequency(result['plumes'])['outside'] for result in eval_results.values()],
    dim='model',
).assign_coords(model=list(eval_results))
outside_df = (outside_da.drop_vars(['lat', 'lon'], errors='ignore').to_series()
              .unstack('season').reindex(columns=list(ev.SEASONS)).round(0))
outside_df['chance'] = [round(200 / (eval_pairs[model][0].sizes['member'] + 1), 1) for model, _ in outside_df.index]
outside_df

**What to look for**
- Raw values outside the plume in most years, with anomalies inside it: a mean-state bias only.
- Anomalies outside the plume *at both extremes* more often than chance: too little noise, or a signal problem (which one is settled in section 5).
- ERA5 drifting from one edge of the anomaly plume towards the other through the record: the trend differs (the signal).

## 4. The moment test, and the other statistics

Each panel is one statistic in one season. Every member's value is a blue dot, the shading is the members' 5–95% range, and ERA5 is the black line and diamond. If the model were perfect, ERA5 would be just another dot. It is flagged (✗) when it falls outside the shading, i.e. p < 0.1 two-sided.

**The moment test** (`ev.moment_test`) compares the first four moments: the mean of the raw values, and the standard deviation, skewness and excess kurtosis of the detrended values. **The other statistics** (`ev.statistic_test`: trend, S/N, width, tails, records, lag-1) are not moments, but are tested in exactly the same way.

Why this is a proper test: every member's statistic is computed from exactly the same years as ERA5's. So the spread of the dots is the sampling distribution of that statistic for this record length, under the model's climate, with no need for a parametric formula or an effective sample size. This is how large-ensemble studies judge whether a model's statistic is acceptable (Suarez-Gutierrez et al., 2021). Skewness and kurtosis use the plain moment estimators, which are biased for short records, but identically so for ERA5 and the members. The higher moments are hard to pin down from ~35 values: for Gaussian data the standard error of the skewness is about √(6/35) ≈ 0.4 and of the excess kurtosis about √(24/35) ≈ 0.8, so only large shape errors can be detected at a single point. Section 7 tests the whole map at once. The S/N against itself follows the drafts' definition (LOWESS change since the start of the record over the std of the residuals), because ERA5 has no hist-nat to compare with. Record counts are a weak test with ~35 years (only ~4 records are expected); part 2 of *Observed change* pools the seasons for a stronger version.

In [ ]:
#(t): Quick look: the moment test in EVAL_SEASON: every member's value (histogram) and ERA5's (black line)
moments_ds = eval_results[EVAL_MODEL]['moments'].sel(season=EVAL_SEASON)
fig, axes = plt.subplots(1, moments_ds.sizes['statistic'], figsize=(18, 3))
for ax, statistic in zip(axes, moments_ds.statistic.values):
    ax.hist(moments_ds['members'].sel(statistic=statistic).dropna('member'), bins=20, color='0.7')
    ax.axvline(float(moments_ds['era5'].sel(statistic=statistic)), color='k')
    ax.set_title(f"{statistic}: percentile {float(moments_ds['percentile'].sel(statistic=statistic)):.0f}")

In [ ]:
#(t): One model: the moment test (mean, standard deviation, skewness, excess kurtosis)
panels = evp.statistic_grid(eval_results[EVAL_MODEL]['moments'],
                            title=f'Moment test: {EVAL_MODEL} {EVAL_EXPERIMENT} vs ERA5 at {POINT_LABEL}')

In [ ]:
#(t): One model: the other signal and noise statistics (the quantities the response figures use), and lag-1 memory
panels = evp.statistic_grid(eval_results[EVAL_MODEL]['statistics'],
                            statistics=ev.SIGNAL_STATISTICS + ev.NOISE_STATISTICS + ('lag1',),
                            title=f'Signal, noise and memory: {EVAL_MODEL} {EVAL_EXPERIMENT} vs ERA5 at {POINT_LABEL}')

In [ ]:
#(t): One model: record highs and lows (with no change at all, 1 + 1/2 + ... + 1/35 ≈ 4.1 of each are expected)
panels = evp.statistic_grid(eval_results[EVAL_MODEL]['statistics'], statistics=ev.RECORD_STATISTICS,
                            title=f'Records: {EVAL_MODEL} {EVAL_EXPERIMENT} vs ERA5 at {POINT_LABEL}')

In [ ]:
#(t): The same numbers as a table (verdict: -1 ERA5 below the members, +1 above, 0 consistent)
(evp.order_seasons(ev.combine_tests(eval_results[EVAL_MODEL]['moments'], eval_results[EVAL_MODEL]['statistics']))
 [['era5', 'lower', 'median', 'upper', 'percentile', 'pvalue', 'verdict']]
 .drop_vars(['lat', 'lon', 'label', 'units'], errors='ignore')
 .to_dataframe().round(3))

## 5. Rank histograms (Suarez-Gutierrez et al., 2021)

**Method**
1. Give ERA5 and every member the same treatment (anomalies, or detrended anomalies).
2. Each year (and season), rank ERA5 among the members: its rank is the number of members below it.
3. Count the ranks from every year into a histogram. If ERA5 behaves like a member, every rank is equally likely and the histogram is flat.
4. With ~35–140 samples, even a perfect model's histogram is not flat. So repeat steps 2–3 treating each member in turn as the observations, ranked against the other N − 1 ("pseudo-observations", or "model as truth"). Their N histograms show how far from flat a perfect model's histogram can be. The dashed lines and grey bars are their 5–95% range.
5. Summarise each histogram and locate ERA5's summary among the pseudo-observations':
   - **dispersion**: the variance of the rank relative to a flat histogram. > 1 is a U shape (ERA5 too often at the edges: spread too small, too little noise); < 1 is a dome (spread too large, too much noise).
   - **outside / below / above**: how often ERA5 is beyond every member (the crosses at the two ends).
   - **mean rank**: ERA5 sitting mostly low (model too warm) or high (model too cold).
   - **rank trend**: ERA5 drifting through the ensemble over the record (model warming faster or slower than ERA5: the **signal**).

**Signal or noise?** Run the test on anomalies (signal + noise) and on detrended anomalies (noise only). A U shape in both means too little noise. A U shape only in anomalies means the forced response is wrong. See the toy examples in section 1.

**A change from the daily prototype (`archive/draft_current_DAILY.ipynb`).** There, ERA5 was ranked against all N members (ranks 0…N) but each pseudo-observation against N − 1 (ranks 0…N − 1), so the two histograms did not line up. Here ERA5 is ranked against each of the N leave-one-out sub-ensembles of N − 1 members and the histograms averaged, so ERA5 and the pseudo-observations are compared like for like. `ev.leave_one_out_counts` does this in closed form: no loop.

In [ ]:
#(t): The test step by step, for one season of one model (anomalies)
eval_ensemble_da, eval_obs_da = eval_pairs[EVAL_MODEL]
season_ensemble_da = eval_ensemble_da.sel(season=EVAL_SEASON)
season_obs_da = eval_obs_da.sel(season=EVAL_SEASON)
anatomy_result_ds = ev.rank_histogram_test(season_ensemble_da, season_obs_da, 'anomaly')
panels = evp.rank_anatomy(
    ev.treat(season_ensemble_da, 'anomaly'), ev.treat(season_obs_da, 'anomaly'), anatomy_result_ds, units=var.units,
    title=f'How the rank-histogram test works: {EVAL_MODEL}, {EVAL_SEASON} anomalies at {POINT_LABEL}',
)

In [ ]:
#(t): Suarez-Gutierrez et al. style histograms, all four seasons pooled: raw values, anomalies, detrended
panels = evp.rank_histogram_row(
    {ev.TREATMENTS[treatment]: result for treatment, result in eval_results[EVAL_MODEL]['ranks'].items()},
    title=f'Rank histograms, all seasons pooled: {EVAL_MODEL} {EVAL_EXPERIMENT} vs ERA5 at {POINT_LABEL}',
)

In [ ]:
#(t): Every model, compact (ranks grouped into 10 bins of rank / N so ensembles of any size line up)
panels = evp.rank_histogram_grid(
    {model: {ev.TREATMENTS[t]: result['ranks'][t] for t in ev.TREATMENTS} for model, result in eval_results.items()},
    title=f'Rank histograms, all seasons pooled: every model vs ERA5 at {POINT_LABEL}',
)

**What to look for**
- **Raw values**: ranks piled at one end are a mean-state bias (the raw verdict is the mean rank, not the dispersion, which a bias makes meaningless).
- **Anomalies vs detrended**: the same U or dome in both columns is a noise problem; a U in anomalies only is a signal problem.
- **Crosses at the ends**: how often ERA5 is beyond every member. Crosses above the dash-dot range mean ERA5 leaves the ensemble too often.

## 6. All models: scorecard and signal-vs-noise diagram

**Scorecard.** Every model (rows), statistic and season (columns). Colour is ERA5's percentile among that model's members: blue means ERA5 below most members, red above. The darkest classes are beyond the members' 5–95%, and outlined cells are flagged (p < 0.1). Rank statistics come from the per-season rank tests (35 samples each, so less powerful than the pooled histograms above).

**Signal-vs-noise diagram.** For each model and season, ERA5's percentile for the **trend** (x: the signal) against the **standard deviation of detrended anomalies** (y: the noise). Inside the grey square the model is consistent on both. Left of it the model's trend is too large, right of it too small. Below it the model is too variable, above it not variable enough.

In [ ]:
#(t): Where ERA5 falls among each model's members: the moments, then the signal and noise statistics, every season
#(c): Pass statistics=None for every statistic, including records and lag-1
SCORECARD_STATISTICS = [*ev.MOMENTS, *ev.SIGNAL_STATISTICS, 'rank_trend', *ev.NOISE_STATISTICS,
                        'dispersion (anomaly)', 'dispersion (detrended)']
panels = evp.scorecard(eval_table_ds, statistics=SCORECARD_STATISTICS,
                       title=f'Where ERA5 falls among each model\'s members ({POINT_LABEL})')

In [ ]:
#(t): Signal (trend) against noise (interannual variability), every model and season
panels = evp.signal_noise_diagram(eval_table_ds,
                                  title=f'Signal and noise: trend vs interannual variability ({POINT_LABEL})')

In [ ]:
#(t): The same with the distribution's width (Q95 - Q05) as the noise
panels = evp.signal_noise_diagram(eval_table_ds, y='width',
                                  title=f'Signal and noise: trend vs distribution width ({POINT_LABEL})')

**Reading the scorecard**

Expect some flags by chance. The scorecard holds several hundred model-statistic-season cells, and at p < 0.1 about one in ten would be flagged even if every model were perfect. Trust patterns (a whole row, a statistic flagged in every season, several models agreeing) over single cells.

## 7. Maps: the moment test at every grid point, and is each model acceptable?

The functions work at every grid point at once, so the moment test maps out cheaply (a few minutes, so the result is saved): `reduce_to_dataset` runs it on every model's historical node (loading one model's 1979–2014 field, ~100 MB, at a time) and stacks the member-free results into one Dataset. The colours are the same as the scorecard: ERA5's percentile among the members, darkest where ERA5 is beyond the 5–95% range. Models with fewer than 20 members cannot flag anything and are left blank.

**Is the model acceptable over the whole map? (the field test)** A perfect model is flagged at about 10% of grid points by chance, and neighbouring points are correlated, so the flagged area of a perfect model varies a lot and speckle means nothing on its own. `ev.field_test` asks whether ERA5 is flagged over *more* of the map than a perfect model would be. It treats each member in turn as ERA5, against the other N − 1 (the pseudo-observations of the rank histograms), and measures the area each is flagged over. The spread of those areas is what a perfect model gives, with the model's own spatial correlation built in. This is the Monte Carlo field-significance test of Livezey & Chen (1983), with the ensemble supplying the null. The top-left corner of each panel gives ERA5's flagged area and the most a perfect model gives 90% of the time; ✗ means ERA5 is flagged over more (p < 0.1): the model is not acceptable for that moment in that season.

Why not the false discovery rate (Wilks, 2016)? With N members the smallest possible p-value is 2 / (N + 1), about 0.04 for 50 members, so the Benjamini–Hochberg procedure at α_FDR = 0.2 could only reject anything if ERA5 were beyond every member over about a fifth of the map. The pseudo-observation test has no such floor.

### Calculate (slow; saves the result)

In [ ]:
%%time
#(t): The moment test at every grid point for every model with EVAL_EXPERIMENT, and the field test on each map
#(c): Members are dropped once tested, so models of any ensemble size stack on one grid
def map_moment_test(ds):
    result = ev.moment_test(*ev.align(ds[VARIABLE].sel(year=EVAL_YEARS).load(), era5_seasonal_da, years=EVAL_YEARS))
    return xr.merge([result, ev.field_test(result)]).drop_dims('member')

eval_map_table_ds = datatree.reduce_to_dataset(
    lesfmip_obs_tree.match(f'*/{EVAL_EXPERIMENT}'), map_moment_test, coords='minimal', compat='override',
).sel(experiment=EVAL_EXPERIMENT, drop=True)
storage.save(eval_map_table_ds, result_path('eval_map_table'))

### Or reopen the saved result

In [ ]:
eval_map_table_ds = storage.open_dataset(result_path('eval_map_table'), load=True)

### Maps

In [ ]:
#(t): What was made, and a quick look: ERA5's percentile among each model's members for the standard deviation
print(eval_map_table_ds)
eval_map_table_ds['percentile'].sel(statistic='std').plot(x='lon', y='lat', row='model', col='season', vmin=0, vmax=100,
                                                          cmap='RdBu_r');

In [ ]:
#(t): ERA5's percentile for each moment, every model and season, with the field test in each panel's corner
for statistic in ev.MOMENTS:
    panels = evp.percentile_maps(eval_map_table_ds, statistic)

In [ ]:
#(t): The field test for every model, moment and season (✗: ERA5 flagged over more of the map than a perfect model gives)
field_verdicts_da = evp.order_seasons(eval_map_table_ds['field_verdict'])
pd.DataFrame({
    (statistic, season): field_verdicts_da.sel(statistic=statistic, season=season).to_series().map({0: '✓', 1: '✗'})
    for statistic in field_verdicts_da['statistic'].values for season in field_verdicts_da['season'].values
}).fillna('– (N < 20)')

In [ ]:
#(t): How many models are consistent with ERA5 (ERA5 inside their members' 5-95%) at each grid point
#(c): Only models with enough members to be tested (N >= 20) are counted, as consistent or not
testable_da = eval_map_table_ds['testable'] == 1
n_eval_models = int(testable_da.any([dim for dim in testable_da.dims if dim != 'model']).sum())
count_cmap, count_norm = rcp.count_colormap(n_eval_models, base_cmap='Blues')
n_consistent_da = ((eval_map_table_ds['verdict'] == 0) & testable_da).sum('model')
panels = maps.polar_grid(
    evp.order_seasons(n_consistent_da),
    row_dim='statistic', col_dim='season',
    levels=np.arange(-0.5, n_eval_models + 1.5), cmap=count_cmap, norm=count_norm,
    label_fmt={'statistic': lambda s: ev.MOMENTS[s].label},
    title='Number of models consistent with ERA5',
    cbar_label="Models with ERA5 inside their members' 5-95% range",
    tag=False,
)
panels.extras['cbar'].set_ticks(range(n_eval_models + 1))

## 8. Where each model is adequate: the spatial evaluation of Suarez-Gutierrez et al. (2021)

The paper's grid-point evaluation (their sections 2.2.2 and 3.2–3.4), at every grid point and season. It does not use the members-as-observations rank histograms of section 5 (the paper keeps those for the global mean). Instead it reduces each point's rank histogram to three numbers and compares them with what a perfect model gives, using fixed thresholds (`ev.spatial_evaluation`):

1. **Anomalies.** Every member and ERA5 minus its own mean over 1979–2013, so a constant offset is gone and the forced change and the variability are left. (The paper uses 1961–1990, which is before ERA5 starts.)
2. **Years.** Only years ERA5 has; a point with fewer than 10 is not tested.
3. **Rank.** Each year, is ERA5 below every member (rank 0), above every member (rank N), or in between?
4. **Three percentages.** The % of years below every member, above every member, and inside the members' central 75% (12.5–87.5%).
5. **Against a perfect model.** If ERA5 were one more member, it would be below every member in 100 / (N + 1) % of years (2% for 50 members), above in as many, and in the central 75% about 75% of the time. Short records push these up by chance, so the paper flags only **≥ 10% below**, **≥ 10% above** or **> 80% in the middle**. Models with N < 20 are not tested with these thresholds: 10 members already put ERA5 beyond them 9% of the time on each side. `SPATIAL_THRESHOLDS = 'perfect_model'` (not in the paper) sets the thresholds from the members instead: each member in turn is treated as ERA5 against the other N − 1, and a percentage counts as a problem above the 95th percentile of theirs. That works for any N.
6. **Diagnosis**, using when the edge years fall (1979–1996 or 1997–2013):

| ERA5 is … | Diagnosis |
|---|---|
| below the members mostly late, or above them mostly early | the model **warms too much** (forced response too strong) |
| above the members mostly late, or below them mostly early | the model **warms too little** |
| both below and above, throughout | **too little variability**: the spread is too narrow |
| in the central 75% more than 80% of the time | **too much variability**: the spread is too wide |
| beyond one edge only, throughout | **one tail too short**: the shape of the distribution |
| none of these | **adequate** |

   The sign flips with time because anomalies are taken over the whole record: a model that warms too fast looks too cold early and too warm late, relative to ERA5. With both edges beyond 10%, both must point the same way to count as the forced response.
7. **Early against late.** A model's variability changes little over time, but the forcing does. So each problem over the whole record is checked for an uneven split between 1979–1996 and 1997–2013 (a binomial test, p < 0.1). Spread evenly, it is the **variability**; concentrated in one period, or flipping between them, the **forced response**. Each half is only ~17 years per season, too short to compare its own flags directly, which in 17 years would flag even a perfect model's middle about a third of the time.
8. **Score.** The adequate share of the tested area per model (each panel's corner), and how many models are adequate at each grid point (the paper's Fig. 8).

### Calculate (slow; saves the result)

In [ ]:
%%time
#(t): The spatial evaluation for every model with EVAL_EXPERIMENT: the whole record, each half, and the period test
SPATIAL_THRESHOLDS = 'fixed'   # the paper's 10% / 10% / 80%; 'perfect_model' sets them from the members (not in the paper)
SPATIAL_PERIODS = {'early': slice(config.ERA5_START_YEAR, 1996), 'late': slice(1997, None)}


def spatial_evaluation(ds):
    ensemble, obs = ev.align(ds[VARIABLE].sel(year=EVAL_YEARS).load(), era5_seasonal_da, years=EVAL_YEARS)
    whole = ev.spatial_evaluation(ensemble, obs, thresholds=SPATIAL_THRESHOLDS)
    early, late = (ev.spatial_evaluation(ensemble, obs, period=period, thresholds=SPATIAL_THRESHOLDS)
                   for period in SPATIAL_PERIODS.values())
    return whole.assign(period_diagnosis=ev.period_comparison(whole, early, late))


spatial_table_ds = datatree.reduce_to_dataset(
    lesfmip_obs_tree.match(f'*/{EVAL_EXPERIMENT}'), spatial_evaluation, coords='minimal', compat='override',
).sel(experiment=EVAL_EXPERIMENT, drop=True)
storage.save(spatial_table_ds, result_path('spatial_table'))

### Or reopen the saved result

In [ ]:
spatial_table_ds = storage.open_dataset(result_path('spatial_table'), load=True)

### Maps

In [ ]:
#(t): The paper's Fig. 5: ERA5 beyond every member (red above, blue below) or too often in their middle (hatched)
panels = evp.spatial_maps(spatial_table_ds)

In [ ]:
#(t): What is wrong at each grid point (step 7)
panels = evp.diagnosis_maps(spatial_table_ds)

In [ ]:
#(t): Each problem early vs late: the same (variability) or changing (forced response)?
panels = evp.diagnosis_maps(spatial_table_ds, variable='period_diagnosis')

In [ ]:
#(t): The score: % of the tested area where each model is adequate, per season
evp.order_seasons(ev.adequate_area(spatial_table_ds)).to_pandas().round(0)

In [ ]:
#(t): The paper's Fig. 8: how many models are adequate at each grid point
panels = evp.adequate_count_maps(ev.adequate_count(spatial_table_ds))

**Reading these maps**
- A model warming too much shows blue (ERA5 below every member) in the late period and red early, and a forced-response verdict for the period split. Too little variability shows as purple (both edges) with a variability verdict.
- Hatching (too much variability) is the least robust signal with ~35 years per season: the paper notes that short periods put the observations in the middle of the ensemble too often by chance.
- Each grid point is tested on ~35 values, so expect some flags by chance; read coherent regions. A regional mean (as in the observed-change section) is the more robust check, as the paper recommends.

## Caveats

- **ERA5 is not the truth.** Over the Antarctic interior, ERA5 near-surface temperature is constrained by few observations, especially early in the record. A model-ERA5 mismatch can be an ERA5 problem.
- **A short record.** ~35 values per season give the tests limited power: "consistent" means "not distinguishable in 35 years", not "correct". Pooling seasons (section 5) quadruples the samples, but assumes all four seasons share the same kind of error.
- **Multiple testing.** Many statistics × seasons × models × grid points are tested at p < 0.1. On the maps, the field test (section 7) accounts for the grid points: it compares the flagged area with what a perfect model gives. Across statistics, seasons and models, still expect about 10% flags by chance, and read coherent patterns rather than single cells.
- **Treatments interact.** Too little noise can also flag the mean and trend (a too-narrow ensemble underestimates how much a 35-year mean or trend can vary), and a bias makes the raw rank dispersion meaningless. Read the noise tests first, and use anomalies for everything except the mean.
- **Ensemble size.** p < 0.1 needs N ≥ 20 members; the member-count table in the setup shows N for each model.

# Part 2: observed change

The ERA5 evaluation asked whether ERA5 could be a member of the *historical* ensemble. Attribution asks more of the observations:
1. **Detection.** Could the observed change have happened without human influence? ERA5 outside the **hist-nat** members' range says no.
2. **Consistency.** Which experiments could have produced it: historical, or a single forcing on its own?
3. **Records.** With no change at all, year *n* of a record is its warmest so far with probability 1/*n*, so 1 + 1/2 + … + 1/*n* records are expected. Warming brings more record highs and fewer record lows.
4. **Scaling factors.** How much of each model's forced response is in the observations: ERA5 ≈ Σ β × response + internal variability. β = 1 means the model's response has the observed size; an interval clear of 0 is a detection.

**Where.** A single grid point has ~35 noisy values per season, so most of this section uses regional means (`oc.REGIONS`), which average away much of the noise. The period is 1979–2014, the overlap of ERA5 and LESFMIP historical.

Calculations live in `extant/observed_change.py` (`oc`) and figures in `extant/plots/observed_change.py` (`ocp`).

In [ ]:
#(t): Settings
OBS_REGION = 'Antarctica (90–65°S)'   # region shown in the single-region figures; see oc.REGIONS
OBS_MODEL = config.PLOT_SEL['model']         # model shown in the single-model figures
OBS_EXPERIMENTS = [e for e in config.MAIN_EXPERIMENTS if all(e in b.children for b in lesfmip_season_tree.children.values())]
oc.REGIONS

In [ ]:
#(t): Regional means of ERA5 and of every member, 1979-2014, for every region
obs_period_tree = lesfmip_obs_tree
regional_trees = {region: oc.regional_mean(obs_period_tree, region).load() for region in oc.REGIONS}
regional_era5 = {region: oc.regional_mean(era5_seasonal_da, region).load() for region in oc.REGIONS}


#(c): Experiment -> the model's members, regionally averaged, on (member, year, season)
def regional_ensembles(region, model, experiments=OBS_EXPERIMENTS):
    return {e: regional_trees[region][model][e][VARIABLE] for e in experiments}

## 1. Which experiments is the observed change consistent with?

Each row is one experiment: its members' 1979–2014 trends (dots) and their 5–95% range (bar). The black line is ERA5's trend. Where the line crosses a row's bar (✓), that experiment could have produced the observed trend; where it misses (✗), it could not. ERA5 outside **hist-nat** is a detection of forced change; ERA5 inside **historical** is consistency. The same figure for the S/N against itself shows whether ERA5 has emerged from its own noise as far as the members have.

In [ ]:
#(t): ERA5's trend, S/N and record counts among each experiment's members, every model, in OBS_REGION
consistency = {model: oc.experiment_consistency(regional_ensembles(OBS_REGION, model), regional_era5[OBS_REGION],
                                                statistics=('trend', 'emergence', 'records_high', 'records_low'))
               for model in lesfmip_season_tree.children}
panels = ocp.experiment_strips(consistency[OBS_MODEL], 'trend',
                               title=f'Observed trend vs each experiment: {OBS_MODEL}, {OBS_REGION}')

In [ ]:
#(t): The same for the S/N against itself (smoothed change over the record / its own noise)
panels = ocp.experiment_strips(consistency[OBS_MODEL], 'emergence',
                               title=f'Observed S/N vs each experiment: {OBS_MODEL}, {OBS_REGION}')

In [ ]:
#(t): Every model: ERA5's trend percentile among each experiment's members (outlined: outside 5-95%)
panels = evp.scorecard(oc.consistency_table(consistency, 'trend'),
                       title=f"ERA5's 1979-2014 trend among each experiment's members ({OBS_REGION})")

## 2. Record rates

Cumulative record highs and lows through 1979–2014, with the four seasons pooled so there are enough records to compare. The dashed curve is what a climate with no change gives (1 + 1/2 + … + 1/*n* per season). Record highs arriving faster than that, and faster than the hist-nat members', mean warming the natural range cannot produce; record lows arriving slower mean the cold end has moved too.

In [ ]:
#(t): Record rates in OBS_REGION: ERA5 against hist-nat and historical
record_curves_ds = oc.record_curves(regional_ensembles(OBS_REGION, OBS_MODEL, ['hist-nat', 'historical']),
                                 regional_era5[OBS_REGION])
panels = ocp.record_rates(record_curves_ds, title=f'Record rates, all seasons pooled: {OBS_MODEL}, {OBS_REGION}')

In [ ]:
#(t): Record rates at the ERA5-evaluation grid point (much noisier than a regional mean)
point_curves_ds = oc.record_curves(
    {e: lesfmip_obs_tree[OBS_MODEL][e][VARIABLE].sel(**EVAL_POINT).load() for e in ('hist-nat', 'historical')},
    era5_seasonal_da.sel(**EVAL_POINT))
panels = ocp.record_rates(point_curves_ds, title=f'Record rates, all seasons pooled: {OBS_MODEL}, {POINT_LABEL}')

## 3. Detection maps

The trend test at every grid point, against hist-nat and against historical, sorted into four classes:
- **detected, consistent**: outside hist-nat's range and inside historical's: the classic attribution result;
- **not detected, consistent**: inside both: the observed trend could be natural, and historical agrees;
- **detected, inconsistent**: outside both: the change is forced, but historical's is too large or too small;
- **not detected, inconsistent**: inside hist-nat but outside historical: the observations look natural, but the model has changed more (or less).

This takes a few minutes, so the result is saved. At p < 0.1 about 10% of points land outside a range by chance, and neighbouring points are correlated, so read coherent regions rather than single points.

### Calculate (slow; saves the result)

In [ ]:
%%time
#(t): Detection classes for the 1979-2014 trend, every model (one model's two experiments loaded at a time)
detections_ds = oc.detection_maps(lesfmip_obs_tree, era5_seasonal_da, variable=VARIABLE)
storage.save(detections_ds, result_path('detections'))

### Or reopen the saved result

In [ ]:
detections_ds = storage.open_dataset(result_path('detections'), load=True)

### Maps

In [ ]:
panels = ocp.detection_maps(detections_ds)

In [ ]:
#(t): How many models call each point detected and consistent
panels = ocp.detection_count_maps(detections_ds)

## 4. Scaling factors: how much of each forced response is in the observations

A regression of ERA5 on each model's forced responses, over every year and season of 1979–2014 (each series measured from its own seasonal means):
- **ANT + NAT**: anthropogenic (historical − hist-nat) and natural (hist-nat);
- **GHG + AER + O3 + NAT**: the single forcings. These are harder to separate: the GHG and aerosol trends are strongly (anti-)correlated over this period, and hist-totalO3 has the fewest members.

The uncertainty comes from a perfect-model test: each historical member in turn plays the observations, regressed on fingerprints that leave it out. The spread of those β's is what internal variability alone does to β, and their median (the grey tick) shows any bias in the method; it should be close to 1. A β below the tick is partly regression dilution from noisy fingerprints (worst for small ensembles), not necessarily a model overestimating the response.

In [ ]:
%%time
#(t): Scaling factors for every region, model and set of signals
scaling = {
    signals: {region: {model: oc.scaling_factors(regional_ensembles(region, model), regional_era5[region], signals)
                       for model in lesfmip_season_tree.children}
              for region in oc.REGIONS}
    for signals in oc.SIGNAL_SETS
}

In [ ]:
#(t): Two signals: anthropogenic and natural
panels = ocp.scaling_factor_plot(scaling['ANT + NAT'], title='Scaling factors: anthropogenic (ANT) and natural (NAT)')
panels = ocp.attributable_trend_plot(scaling['ANT + NAT'], title='How much of the observed 1979-2014 trend ANT and NAT explain')

In [ ]:
#(t): Four signals: greenhouse gases, aerosols, ozone and natural
panels = ocp.scaling_factor_plot(scaling['GHG + AER + O3 + NAT'], title='Scaling factors: single forcings')
panels = ocp.attributable_trend_plot(scaling['GHG + AER + O3 + NAT'], title='How much of the observed trend each forcing explains')

In [ ]:
#(t): The numbers: beta, its 5-95% interval, detected (interval clear of 0) and consistent (interval includes 1)
pd.concat({
    (signals, region, model): result[['beta', 'lower', 'upper', 'detected', 'consistent', 'perfect_model_beta']]
    .to_dataframe()
    for signals, by_region in scaling.items() for region, by_model in by_region.items()
    for model, result in by_model.items()
}, names=['signals', 'region', 'model']).round(2)

**Caveats**
- **Short record.** 1979–2014 gives ~35 values per season, so detection is hard for weak signals, and "not detected" is not "no effect".
- **ERA5 over the Antarctic interior** is constrained by few observations, especially early on; an inconsistency can be an ERA5 problem.
- **Regression.** Ordinary least squares with noisy fingerprints biases β low (regression dilution); the perfect-model median shows by how much. Strongly correlated fingerprints (GHG and aerosols) give wide, anti-correlated intervals, so trust the two-signal result more.
- **Independence.** The perfect-model spread assumes the historical members' internal variability is realistic, which the ERA5 evaluation (noise tests) checks.